In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [5]:
items = pd.read_csv('item_list2.txt', sep=' ')
items.head()

,org_id,remap_id
0,1881509818,1
1,2094869245,2
2,7245456259,3
3,7245456313,4
4,B000002NUS,5


In [80]:
items_set = set(items['org_id'])

with open('meta_Sports_and_Outdoors.json', 'r') as f:
    all_lines_len = len(f.readlines())
all_lines_len

532197

In [25]:
import ast

data = []
with open('meta_Sports_and_Outdoors.json', 'r') as f:
    i = 0
    for line in f:
        candidate = dict(ast.literal_eval(line))
        if candidate['asin'] in items_set:
            data.append(candidate)
        # print(candidate['asin'], type(candidate['asin']))
        if i % int(all_lines_len * 0.05) == 0:
            print(round(i / all_lines_len * 100, 2))
        i+=1
        # break
print(data[0])

0.0
5.0
10.0
15.0
20.0
25.0
30.0
35.0
40.0
45.0
50.0
55.0
60.0
65.0
70.0
75.0
80.0
85.0
90.0
95.0
100.0
{'asin': '1881509818', 'related': {'also_bought': ['B000U3YWEM', 'B000U401J6', 'B004JLY2GE', 'B004NGP8X6', 'B003WG95H8', 'B006MYLE9O', 'B0065PJY60', 'B008VYYD6Y', 'B00A1WEMRE', 'B004Z0LQYA', 'B0081JJVUC', 'B001AT6S7O', 'B00FGL97CA', 'B000J4HN9I', 'B004ERKCIA', 'B000O5ILUM', 'B005PC92BG', 'B0098TBVW0', 'B000EU02S6', 'B002IY5BZ0', 'B0057IO0QA', 'B0077AXZSA', 'B0048KGFHU', 'B001T7QJ9O', 'B00555BFDG', 'B00162SAQ2', 'B0062CB360', 'B000HBPQLU', 'B0047WKF84', 'B005MZU7DS', 'B00162OKDY', 'B005VNWMRU', 'B003E0OWKW', 'B001AT1GJE', 'B006T6Y56E', 'B004VIZN90', 'B0086UBC3A', 'B000NJY1YO', 'B005N035WM', 'B005N00DTK', 'B00DPKZ1MO', 'B004DPGM1O', 'B001HBHNHE', 'B005G2H3QQ', 'B003JW4JIA', 'B003WG0PB8', 'B00FDWM9OK', 'B0064Q0H2U', 'B000QSJQTW', 'B0002IKANW', 'B00A6VX9M4', 'B001ASZSVW', 'B006X38ISY', 'B004WRWAQO', 'B004VJ02OA', 'B007T0SDJE', 'B00AUC2DEE', 'B000NK0DJA', 'B00162QGLS', 'B0000C50HM', 'B00J

In [59]:
items_and_categories_lst = [
    [row["asin"], row["categories"], len(row["categories"])] for row in data
]

In [102]:
items_and_categories_df = pd.DataFrame(
    items_and_categories_lst, columns=["Item_id", "Categories", "Categories Length"]
)

In [107]:
embeddings = pd.read_pickle('embeddings2.pkl')

print("Embeddings shape:", embeddings.shape)
print("Items shape:", items_and_categories_df.shape)

embeddings = embeddings[1:]

print("Embeddings shape after removed first row:", embeddings.shape)
print("Items shape:", items_and_categories_df.shape)

items_and_categories_df['embeddings'] = list(embeddings)
print("New Items shape:", items_and_categories_df.shape)

Embeddings shape: (18358, 256)
Items shape: (18357, 4)
Embeddings shape after removed first row: (18357, 256)
Items shape: (18357, 4)
New Items shape: (18357, 4)


In [175]:
X = items_and_categories_df.drop(columns=['Categories'])['embeddings']
y = items_and_categories_df['Categories'].map(lambda row: row[0])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

In [ ]:
y_subcategories_set = set()


,Accessories,Accessories & Supplies,Accessory Kits,Action Sports,Adjustable Benches,Adult Helmets,Agility Ladders,Air Decompression Limit Monitors,Air Filter Accessories & Cleaning Products,Air Filters & Accessories,...,Women's Balls,Workout Shorts,Workstands,Wraps,Wrestling,Wrist Guards,Wrist Weights,Wristbands,Yoga,Youth Bow Sets
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [180]:
y.values

array([list(['Sports & Outdoors', 'Hunting & Fishing', 'Hunting', 'Gun Maintenance', 'Gunsmithing Tools']),
       list(['Sports & Outdoors', 'Cycling', 'Lights & Reflectors', 'Taillights']),
       list(['Sports & Outdoors', 'Exercise & Fitness', 'Accessories', 'Exercise Bands']),
       ...,
       list(['Sports & Outdoors', 'Outdoor Gear', 'Camping & Hiking', 'Camping Furniture', 'Cots & Hammocks']),
       list(['Sports & Outdoors', 'Boating & Water Sports', 'Boating', 'Dry Bags']),
       list(['Sports & Outdoors', 'Accessories', 'Sports Water Bottles'])],
      shape=(18357,), dtype=object)

In [160]:
X_train_np, X_test_np = np.stack(X_train.values), np.stack(X_test.values)
y_train_np, y_test_np = y_train.values, y_test.values

In [161]:
print(X_train_np.shape, X_test_np.shape)
print(y_train_np.shape, y_test_np.shape)

(14685, 256) (3672, 256)
(14685,) (3672,)


In [174]:
y_train_np

array(['Sports & Outdoors', 'Sports & Outdoors', 'Sports & Outdoors', ...,
       'Sports & Outdoors', 'Sports & Outdoors', 'Sports & Outdoors'],
      shape=(14685,), dtype=object)

In [163]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

for k in [2, 4, 8, 16, 32, 64]:
    model = KNeighborsClassifier(n_neighbors=k, metric='cosine')
    model.fit(X_train_np, y_train_np)
    y_pred = model.predict(X_test_np)
    print(classification_report(y_test_np, y_pred, zero_division=0))

                           precision    recall  f1-score   support

               Automotive       0.00      0.00      0.00         6
              CDs & Vinyl       0.00      0.00      0.00         1
Cell Phones & Accessories       0.00      0.00      0.00         4
              Electronics       0.00      0.00      0.00         1
              Movies & TV       0.00      0.00      0.00         0
     Patio, Lawn & Garden       0.00      0.00      0.00         1
        Sports & Outdoors       1.00      1.00      1.00      3657
 Tools & Home Improvement       0.00      0.00      0.00         2

                 accuracy                           0.99      3672
                macro avg       0.12      0.12      0.12      3672
             weighted avg       0.99      0.99      0.99      3672

                           precision    recall  f1-score   support

               Automotive       0.00      0.00      0.00         6
              CDs & Vinyl       0.00      0.00      0.00  